# Weather Impact Assessment Agent

An agentic system that pulls forecasts and alerts from the National Weather
Service, combines them with NOAA storm history, and assesses the impact on
business locations — all orchestrated through OpenAI function calling.

## Why an Agent?

Weather-driven business decisions require **contextual reasoning**: a tornado
watch matters very differently for a warehouse in Oklahoma versus a data center
in Oregon. An agent can inspect the forecast, check whether active alerts
overlap with *your* locations, pull historical storm data for calibration, and
then synthesize an actionable recommendation — choosing which tools to call
and in what order depending on what it discovers at each step.

In [ ]:
%pip install openai requests pandas --quiet

In [ ]:
import json, textwrap, time
import requests
import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## Sample Business Locations

Each site has a lat/lon, state code, and a short description of what it does
and how critical it is.

In [ ]:
BUSINESS_LOCATIONS = {
    "hq-dallas": {
        "name": "Corporate HQ",
        "lat": 32.78,
        "lon": -96.80,
        "state": "TX",
        "type": "office",
        "criticality": "high",
        "employees": 450,
    },
    "warehouse-okc": {
        "name": "Central Warehouse",
        "lat": 35.47,
        "lon": -97.52,
        "state": "OK",
        "type": "warehouse",
        "criticality": "critical",
        "employees": 120,
    },
    "dc-ashburn": {
        "name": "Primary Data Center",
        "lat": 39.04,
        "lon": -77.49,
        "state": "VA",
        "type": "data_center",
        "criticality": "critical",
        "employees": 35,
    },
    "factory-houston": {
        "name": "Manufacturing Plant",
        "lat": 29.76,
        "lon": -95.37,
        "state": "TX",
        "type": "factory",
        "criticality": "critical",
        "employees": 300,
    },
    "office-denver": {
        "name": "Regional Office",
        "lat": 39.74,
        "lon": -104.99,
        "state": "CO",
        "type": "office",
        "criticality": "medium",
        "employees": 80,
    },
}

pd.DataFrame.from_dict(BUSINESS_LOCATIONS, orient="index")

## Tool 1 — `get_forecast`

Uses the NWS API to fetch a 7-day forecast for a given lat/lon.

In [ ]:
NWS_HEADERS = {"User-Agent": "(weather-impact-agent, contact@example.com)"}


def get_forecast(lat: float, lon: float) -> str:
    """Get the NWS 7-day forecast for a latitude/longitude pair."""
    try:
        # Step 1: resolve grid point
        point_url = f"https://api.weather.gov/points/{lat},{lon}"
        pt = requests.get(point_url, headers=NWS_HEADERS, timeout=15).json()
        forecast_url = pt["properties"]["forecast"]

        # Step 2: get forecast
        fc = requests.get(forecast_url, headers=NWS_HEADERS, timeout=15).json()
        periods = fc["properties"]["periods"][:6]  # next 3 days (day+night)
        result = [
            {
                "name": p["name"],
                "temperature": p["temperature"],
                "temperatureUnit": p["temperatureUnit"],
                "windSpeed": p["windSpeed"],
                "shortForecast": p["shortForecast"],
                "detailedForecast": p["detailedForecast"][:200],
            }
            for p in periods
        ]
        return json.dumps(result, indent=2)
    except Exception as exc:
        return json.dumps({"error": str(exc)})

## Tool 2 — `get_alerts`

Fetches active weather alerts for a US state from the NWS API.

In [ ]:
def get_alerts(state: str) -> str:
    """Get active NWS alerts for a US state (two-letter code, e.g. 'TX')."""
    try:
        url = f"https://api.weather.gov/alerts/active?area={state.upper()}"
        resp = requests.get(url, headers=NWS_HEADERS, timeout=15)
        resp.raise_for_status()
        features = resp.json().get("features", [])
        alerts = []
        for f in features[:10]:  # cap at 10
            props = f["properties"]
            alerts.append({
                "event": props.get("event"),
                "severity": props.get("severity"),
                "headline": props.get("headline", "")[:200],
                "areaDesc": props.get("areaDesc", "")[:200],
                "onset": props.get("onset"),
                "expires": props.get("expires"),
            })
        return json.dumps(alerts, indent=2) if alerts else json.dumps([])
    except Exception as exc:
        return json.dumps({"error": str(exc)})

## Tool 3 — `get_storm_history`

Downloads NOAA Storm Events CSV data (bulk files on FTP) for a given state
and year, returning a summary of event types and counts.

In [ ]:
import io, gzip

def get_storm_history(state: str, year: int = 2023) -> str:
    """Fetch NOAA Storm Events summary for a state and year. Returns event
    type counts and notable incidents."""
    state = state.upper()
    # NOAA bulk CSV files follow this pattern
    url = (
        f"https://www1.ncdc.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
        f"StormEvents_details-ftp_v1.0_d{year}_c20240620.csv.gz"
    )
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        with gzip.open(io.BytesIO(resp.content), "rt") as fh:
            df = pd.read_csv(fh, low_memory=False)

        df_state = df[df["STATE"].str.upper() == state]
        if df_state.empty:
            return json.dumps({"state": state, "year": year, "events": [], "note": "No events found"})

        summary = (
            df_state.groupby("EVENT_TYPE")
            .agg(count=("EVENT_ID", "count"))
            .sort_values("count", ascending=False)
            .head(10)
        )
        return json.dumps(
            {"state": state, "year": year, "top_events": summary.reset_index().to_dict(orient="records")},
            indent=2,
        )
    except Exception as exc:
        # Fallback: return synthetic summary when the bulk file URL changes
        fallback = {
            "state": state,
            "year": year,
            "note": f"Live fetch failed ({exc}); using synthetic summary.",
            "top_events": [
                {"EVENT_TYPE": "Thunderstorm Wind", "count": 312},
                {"EVENT_TYPE": "Hail", "count": 275},
                {"EVENT_TYPE": "Flash Flood", "count": 98},
                {"EVENT_TYPE": "Tornado", "count": 62},
                {"EVENT_TYPE": "Winter Storm", "count": 34},
            ],
        }
        return json.dumps(fallback, indent=2)

## Tool 4 — `assess_business_impact`

A local function that combines weather data with site metadata to estimate
business impact.

In [ ]:
def assess_business_impact(weather_data: dict, assets: dict | None = None) -> str:
    """Assess business impact given weather conditions and business assets.

    *weather_data* should have keys: site_id, alert_severity (None|'Minor'|'Moderate'|'Severe'|'Extreme'),
    wind_speed_mph (int), flood_risk (bool), temperature_f (int).
    """
    assets = assets or BUSINESS_LOCATIONS
    site_id = weather_data.get("site_id", "")
    site = assets.get(site_id, {})
    if not site:
        return json.dumps({"error": f"Site {site_id} not found in inventory"})

    sev_score = {None: 0, "Minor": 1, "Moderate": 2, "Severe": 3, "Extreme": 4}
    crit_mult = {"low": 1, "medium": 1.5, "high": 2, "critical": 3}

    alert_sev = weather_data.get("alert_severity")
    wind = weather_data.get("wind_speed_mph", 0)
    flood = weather_data.get("flood_risk", False)

    base = sev_score.get(alert_sev, 0)
    if wind > 60:
        base += 2
    elif wind > 40:
        base += 1
    if flood:
        base += 1

    impact = round(base * crit_mult.get(site.get("criticality", "medium"), 1.5), 1)

    recommendations = []
    if impact >= 6:
        recommendations.append("ACTIVATE business continuity plan")
        recommendations.append("Consider temporary site shutdown")
    elif impact >= 3:
        recommendations.append("Put incident response team on standby")
        recommendations.append("Pre-position emergency supplies")
    else:
        recommendations.append("Monitor situation; no immediate action required")

    return json.dumps(
        {
            "site_id": site_id,
            "site_name": site.get("name"),
            "impact_score": impact,
            "impact_level": "critical" if impact >= 6 else "elevated" if impact >= 3 else "normal",
            "recommendations": recommendations,
        },
        indent=2,
    )

## Tool Schema for OpenAI Function Calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_forecast",
            "description": "Fetch a 3-day NWS forecast for a latitude/longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "lat": {"type": "number", "description": "Latitude"},
                    "lon": {"type": "number", "description": "Longitude"},
                },
                "required": ["lat", "lon"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_alerts",
            "description": "Get active weather alerts for a US state (two-letter code).",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {"type": "string", "description": "Two-letter state code, e.g. TX"},
                },
                "required": ["state"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_storm_history",
            "description": "Fetch NOAA Storm Events summary for a US state and year.",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {"type": "string", "description": "Two-letter state code"},
                    "year": {"type": "integer", "description": "Year (e.g. 2023)"},
                },
                "required": ["state"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "assess_business_impact",
            "description": "Assess business impact of weather conditions on a specific site. Provide site_id, alert_severity, wind_speed_mph, flood_risk, and temperature_f.",
            "parameters": {
                "type": "object",
                "properties": {
                    "weather_data": {
                        "type": "object",
                        "properties": {
                            "site_id": {"type": "string"},
                            "alert_severity": {"type": "string", "nullable": true},
                            "wind_speed_mph": {"type": "integer"},
                            "flood_risk": {"type": "boolean"},
                            "temperature_f": {"type": "integer"},
                        },
                        "required": ["site_id"],
                    }
                },
                "required": ["weather_data"],
            },
        },
    }
]

## Dispatcher

In [ ]:
TOOL_MAP = {
    "get_forecast": get_forecast,
    "get_alerts": get_alerts,
    "get_storm_history": get_storm_history,
    "assess_business_impact": assess_business_impact,
}

## Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a Weather Impact Assessment agent for a multi-site business.
    Your job:
    1. Check forecasts for the company's locations.
    2. Retrieve active weather alerts for relevant states.
    3. Optionally pull historical storm data for context.
    4. Assess business impact for each site that may be affected.
    5. Produce a concise executive summary with recommended actions.

    Available business sites:
    - hq-dallas (TX, 32.78/-96.80) — Corporate HQ, high criticality
    - warehouse-okc (OK, 35.47/-97.52) — Central Warehouse, critical
    - dc-ashburn (VA, 39.04/-77.49) — Data Center, critical
    - factory-houston (TX, 29.76/-95.37) — Manufacturing, critical
    - office-denver (CO, 39.74/-104.99) — Regional Office, medium

    Use the tools to gather real data before forming your assessment.
""")


def run_agent(user_message: str, max_turns: int = 12) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"\n--- Agent turn {turn + 1} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print("\n--- Agent finished ---")
            return msg.content

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  Calling {fn_name}({fn_args})")
            fn = TOOL_MAP.get(fn_name)
            result = fn(**fn_args) if fn else json.dumps({"error": f"Unknown tool: {fn_name}"})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Agent reached maximum turns without finishing."

## Example Execution

Ask the agent for a full weather impact assessment across all sites.

In [ ]:
report = run_agent(
    "I need a full weather impact assessment for all our business locations. "
    "Check the forecasts, look for active alerts in TX, OK, VA, and CO, and "
    "pull storm history for Oklahoma. For any site with active severe weather, "
    "run the business impact assessment. Give me an executive summary."
)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Second Scenario — Hurricane Season Prep

A more focused query about Gulf Coast exposure.

In [ ]:
report2 = run_agent(
    "Hurricane season is approaching. Focus on our Houston factory and Dallas HQ. "
    "Get the forecasts, check for active Texas alerts, pull Texas storm history for 2023, "
    "and assess business impact assuming Moderate alert severity and 45 mph winds with flood risk. "
    "Give me a readiness recommendation."
)

In [ ]:
Markdown(report2)

## Results Analysis

| Aspect | Detail |
|--------|--------|
| **Data sources** | NWS API (live forecasts + alerts), NOAA Storm Events bulk CSV |
| **Adaptive flow** | Agent skipped impact assessment for sites with clear skies, focused on threatened locations |
| **Impact scoring** | Combines alert severity, wind speed, flood risk with site criticality |
| **Limitations** | NWS API only covers US territories; NOAA CSV URL changes yearly; impact model is simplified |

### Potential Extensions

1. **Geofencing** — use NWS zone polygons to check if a specific alert covers a site's exact coordinates.
2. **Supply chain overlay** — combine with the supply-chain agent to assess logistics disruptions.
3. **Automated alerts** — add a `send_notification` tool for email/SMS when impact score exceeds threshold.
4. **Insurance data** — add a tool to estimate potential loss from historical claim data.

---
*Notebook by the LLM Course — Agentic Designs series*